In [2]:
import os
import json
from pathlib import Path
from datetime import datetime
import numpy as np

MODEL_TIER = "2b"
JSONL_OUTPUT_PATH = "/content/drive/MyDrive/vlm-finetuning-project1/results/baseline_test_full_2b_second/predictions.jsonl"

print(f"Target file: {JSONL_OUTPUT_PATH}")
assert os.path.exists(JSONL_OUTPUT_PATH), "File does not exist yet!"

file_stats = os.stat(JSONL_OUTPUT_PATH)
print(f"File Size: {file_stats.st_size / (1024*1024):.2f} MB")
print(f"Last Modified: {datetime.fromtimestamp(file_stats.st_mtime).strftime('%Y-%m-%d %H:%M:%S')}")

records = []
malformed_lines = []

with open(JSONL_OUTPUT_PATH, "r", encoding="utf-8") as f:
    for line_idx, line in enumerate(f):
        line = line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
            records.append((line_idx, rec))
        except json.JSONDecodeError:
            malformed_lines.append(line_idx)

print(f"Total lines parsed: {len(records) + len(malformed_lines)}")
print(f"Valid JSON rows:    {len(records)}")
print(f"Malformed rows:     {len(malformed_lines)}  -> indices: {malformed_lines[:20]}{' ...' if len(malformed_lines)>20 else ''}")

failed_indices = []          # empty raw_output
missing_fence_indices = []   # non-empty but no ```json fence
success_indices = []         # non-empty AND fenced
latencies = []
image_id_by_idx = {}

for line_idx, rec in records:
    raw_output = rec.get("raw_output", "")
    image_id_by_idx[line_idx] = rec.get("image_id", "?")

    if not raw_output.strip():
        failed_indices.append(line_idx)
        continue

    if "```json" not in raw_output:
        missing_fence_indices.append(line_idx)
    else:
        success_indices.append(line_idx)

    lat = rec.get("latency_seconds", 0.0)
    if lat > 0:
        latencies.append(lat)

print(f"✅ Clean successes:         {len(success_indices)}")
print(f"⚠️ Missing ```json fence:   {len(missing_fence_indices)}")
print(f"❌ Empty (crashed) outputs: {len(failed_indices)}")


def find_continuous_ranges(indices_list):
    """Turns [5,6,7,20,21,50] into [(5,7), (20,21), (50,50)]"""
    if not indices_list:
        return []
    idxs = sorted(indices_list)
    ranges = []
    start = prev = idxs[0]
    for i in idxs[1:]:
        if i == prev + 1:
            prev = i
        else:
            ranges.append((start, prev))
            start = prev = i
    ranges.append((start, prev))
    return ranges

def print_ranges(name, indices_list, batch_size_guess=None):
    ranges = find_continuous_ranges(indices_list)
    print(f"\n{name}: {len(indices_list)} total, {len(ranges)} contiguous block(s)")
    for (s, e) in ranges:
        width = e - s + 1
        note = f" (~{width // batch_size_guess} batch(es) @ size {batch_size_guess})" if batch_size_guess else ""
        print(f"   [{s} -> {e}]  width={width}{note}")

print("=== FAILURE RANGE BREAKDOWN ===")
print_ranges("Empty/crashed records", failed_indices, batch_size_guess=32)
print_ranges("Malformed JSON lines", malformed_lines)
print_ranges("Missing ```json fence (successful but unformatted)", missing_fence_indices)

print("="*100)

total = len(records) + len(malformed_lines)
summary = {
    "total_lines": total,
    "malformed_json": len(malformed_lines),
    "empty_failed": len(failed_indices),
    "missing_fence": len(missing_fence_indices),
    "clean_success": len(success_indices),
    "success_rate_pct": round(len(success_indices) / total * 100, 2) if total else 0,
}
print("=== FINAL SUMMARY ===")
for k, v in summary.items():
    print(f"{k.ljust(20)}: {v}")

Target file: /content/drive/MyDrive/vlm-finetuning-project1/results/baseline_test_full_2b_second/predictions.jsonl
File Size: 4.95 MB
Last Modified: 2026-07-19 10:13:46
Total lines parsed: 3004
Valid JSON rows:    3004
Malformed rows:     0  -> indices: []
✅ Clean successes:         3004
⚠️ Missing ```json fence:   0
❌ Empty (crashed) outputs: 0
=== FAILURE RANGE BREAKDOWN ===

Empty/crashed records: 0 total, 0 contiguous block(s)

Malformed JSON lines: 0 total, 0 contiguous block(s)

Missing ```json fence (successful but unformatted): 0 total, 0 contiguous block(s)
=== FINAL SUMMARY ===
total_lines         : 3004
malformed_json      : 0
empty_failed        : 0
missing_fence       : 0
clean_success       : 3004
success_rate_pct    : 100.0


In [3]:
import json
import re

RESULTS_FILE_PATH = "/content/drive/MyDrive/vlm-finetuning-project1/results/baseline_test_full_2b_second/predictions.jsonl"

def strip_fences(text: str) -> str:
    """Strips markdown code fences (e.g., ```json ... ```) from a string."""
    match = re.search(r"```(?:json)?(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()

    # If there's a starting fence but no closing fence (truncated), grab everything after it
    match_open = re.search(r"```(?:json)?\s*(.*)", text, flags=re.DOTALL | re.IGNORECASE)
    if match_open:
        return match_open.group(1).strip()

    return text.strip()

print("Setup complete.")

Setup complete.


In [4]:
total_samples = 0
valid_json_count = 0
invalid_json_count = 0

truncated_candidates = []
other_errors = []

# Typical max token length is 1000. Depending on the tokenizer,
# 1000 tokens translates to roughly 3000-4500 characters.
# We'll use character length as a proxy for truncation analysis.
TRUNCATION_CHAR_THRESHOLD = 3500

with open(RESULTS_FILE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        total_samples += 1
        record = json.loads(line)
        raw_output = record.get("raw_output", "")
        image_id = record.get("image_id", "unknown")

        # Strip markdown fences
        clean_text = strip_fences(raw_output)

        try:
            # Try to parse the JSON
            parsed = json.loads(clean_text)
            valid_json_count += 1
        except json.JSONDecodeError as e:
            invalid_json_count += 1

            # Check for truncation
            char_length = len(raw_output)
            is_unclosed = clean_text.count("{") > clean_text.count("}") or clean_text.count("[") > clean_text.count("]")

            error_info = {
                "image_id": image_id,
                "error": str(e),
                "char_length": char_length,
                "raw_tail": raw_output[-100:], # Look at the last 100 chars
                "is_unclosed_brackets": is_unclosed
            }

            # If the string is very long OR it has unclosed brackets, it's highly likely truncated
            if char_length > TRUNCATION_CHAR_THRESHOLD or is_unclosed:
                truncated_candidates.append(error_info)
            else:
                other_errors.append(error_info)

print(f"Total Samples Processed: {total_samples}")
print(f"Valid JSON: {valid_json_count}")
print(f"Invalid JSON: {invalid_json_count}")
print("-" * 30)
print(f"Likely Truncated (hit max_new_tokens limit): {len(truncated_candidates)}")
print(f"Other Parse Errors (formatting, hallucinated syntax): {len(other_errors)}")

Total Samples Processed: 3004
Valid JSON: 2902
Invalid JSON: 102
------------------------------
Likely Truncated (hit max_new_tokens limit): 94
Other Parse Errors (formatting, hallucinated syntax): 8


In [5]:
# Let's look at a few examples of the truncated outputs to confirm the symptom
num_to_inspect = min(5, len(truncated_candidates))

print(f"--- Inspecting {num_to_inspect} Truncated Outputs ---")
for i in range(num_to_inspect):
    ex = truncated_candidates[i]
    print(f"\nImage ID: {ex['image_id']}")
    print(f"Character Length: {ex['char_length']} (Proxy for tokens)")
    print(f"Parse Error: {ex['error']}")
    print(f"Has unclosed brackets: {ex['is_unclosed_brackets']}")
    print(f"End of generated string (last 100 chars):")
    print(f"...{ex['raw_tail']}")
    print("-" * 50)

--- Inspecting 5 Truncated Outputs ---

Image ID: 0001850
Character Length: 1616 (Proxy for tokens)
Parse Error: Expecting ',' delimiter: line 42 column 20 (char 1608)
Has unclosed brackets: True
End of generated string (last 100 chars):
...1, 1248, 1224, 1258,
      1224, 1258, 1237, 1268,
      1237, 1268, 1250, 1278,
      1250, 1278, 1
--------------------------------------------------

Image ID: 0000938
Character Length: 1699 (Proxy for tokens)
Parse Error: Expecting value: line 52 column 26 (char 1690)
Has unclosed brackets: True
End of generated string (last 100 chars):
...700, 1240, 720, 1260,
    720, 1260, 740, 1280,
    740, 1280, 760, 1300,
    760, 1300, 780, 1320,

--------------------------------------------------

Image ID: 0002594
Character Length: 1659 (Proxy for tokens)
Parse Error: Expecting value: line 45 column 28 (char 1644)
Has unclosed brackets: True
End of generated string (last 100 chars):
...618, 1309,
      600, 999, 618, 1329,
      600, 1009, 618, 1349,
 

In [6]:
num_to_inspect = min(5, len(other_errors))

print(f"--- Inspecting {num_to_inspect} Other errors Outputs ---")
for i in range(num_to_inspect):
    ex = other_errors[i]
    print(f"\nImage ID: {ex['image_id']}")
    print(f"Character Length: {ex['char_length']} (Proxy for tokens)")
    print(f"Parse Error: {ex['error']}")
    print(f"Has unclosed brackets: {ex['is_unclosed_brackets']}")
    print(f"End of generated string (last 100 chars):")
    print(f"...{ex['raw_tail']}")
    print("-" * 50)

--- Inspecting 5 Other errors Outputs ---

Image ID: 0003948
Character Length: 713 (Proxy for tokens)
Parse Error: Invalid control character at: line 5 column 24 (char 350)
Has unclosed brackets: False
End of generated string (last 100 chars):
...
  "excavator": [],
  "rebar": [],
  "worker_with_white_hard_hat": [
    "988,690,1000,700
  ]
}
```
--------------------------------------------------

Image ID: 0003478
Character Length: 827 (Proxy for tokens)
Parse Error: Invalid control character at: line 5 column 25 (char 487)
Has unclosed brackets: False
End of generated string (last 100 chars):
...null,
  "rule_3_violation": null,
  "rule_4_violation": null,
  "excavator": [],
  "rebar": []
}
```
--------------------------------------------------

Image ID: 0003663
Character Length: 791 (Proxy for tokens)
Parse Error: Invalid control character at: line 5 column 24 (char 431)
Has unclosed brackets: False
End of generated string (last 100 chars):
...
  "excavator": [],
  "rebar": [],
  "w

In [ ]:
import numpy as np
from transformers import AutoTokenizer

# If you already have the tokenizer loaded in your notebook (e.g., from Unsloth),
# you can use it directly. Otherwise, we load it here.
# Replace "unsloth/Qwen2-VL-2B-Instruct" with the actual model path you used.
import numpy as np
from transformers import AutoTokenizer

# Set your Hugging Face Token here

try:
    _ = tokenizer  # Check if tokenizer exists in memory
    print("Using already loaded tokenizer.")
except NameError:
    print("Loading tokenizer...")
    # PASS THE TOKEN HERE
    tokenizer = AutoTokenizer.from_pretrained(
        "unsloth/Qwen3-VL-2B-Instruct",
        token=HF_TOKEN
    )

print("\nCalculating true token counts for truncated outputs...")

token_counts = []
hit_max_tokens_count = 0
MAX_TOKENS_LIMIT = 1000  # The budget you set during inference

for ex in truncated_candidates:
    # Find the corresponding original record from the JSONL
    # We kept the raw_tail and error, but need the full raw string.
    # Alternatively, we could have saved raw_output in truncated_candidates.
    pass

# Let's re-scan the file specifically for the failed JSONs to get the true token counts,
# since we didn't save the full raw_output in the previous cell's list to save memory.
import json

true_truncation_stats = []

with open(RESULTS_FILE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        record = json.loads(line)
        raw_output = record.get("raw_output", "")
        image_id = record.get("image_id", "unknown")

        # Strip markdown fences
        clean_text = strip_fences(raw_output)

        try:
            json.loads(clean_text)
        except json.JSONDecodeError:
            # It's an invalid JSON. Let's count its tokens.
            # return_length=True gives us the token length directly without returning the tensors
            tokens = tokenizer(raw_output, return_length=True)
            token_length = tokens["length"][0] if isinstance(tokens["length"], list) else tokens["length"]

            is_unclosed = clean_text.count("{") > clean_text.count("}") or clean_text.count("[") > clean_text.count("]")

            true_truncation_stats.append({
                "image_id": image_id,
                "token_length": token_length,
                "char_length": len(raw_output),
                "is_unclosed": is_unclosed
            })

            if token_length >= MAX_TOKENS_LIMIT:
                hit_max_tokens_count += 1
            token_counts.append(token_length)

# Reporting
if not token_counts:
    print("No invalid JSONs found to analyze.")
else:
    avg_tokens = np.mean(token_counts)
    max_tokens = np.max(token_counts)

    print("-" * 50)
    print("TOKEN COUNT ANALYSIS FOR FAILED JSONS:")
    print("-" * 50)
    print(f"Total Failed JSONs Analyzed: {len(token_counts)}")
    print(f"Number of outputs hitting the {MAX_TOKENS_LIMIT} token limit: {hit_max_tokens_count}")
    print(f"Average tokens per failed output: {avg_tokens:.1f}")
    print(f"Max tokens seen: {max_tokens}")
    print("-" * 50)

    # Calculate correlation/ratio between chars and tokens
    chars = [s["char_length"] for s in true_truncation_stats]
    avg_chars_per_token = np.mean(chars) / avg_tokens if avg_tokens > 0 else 0
    print(f"Observed Character-to-Token ratio for these failures: {avg_chars_per_token:.2f} chars/token")
    print("*(If this ratio is low, e.g., < 2.5, it proves numeric arrays tokenize highly inefficiently)*")

    print("\nTop 5 longest outputs (by token count):")
    sorted_stats = sorted(true_truncation_stats, key=lambda x: x["token_length"], reverse=True)
    for i, stat in enumerate(sorted_stats[:5]):
        print(f"{i+1}. Image ID: {stat['image_id']} | Tokens: {stat['token_length']} | Chars: {stat['char_length']} | Unclosed Brackets: {stat['is_unclosed']}")

Using already loaded tokenizer.

Calculating true token counts for truncated outputs...
--------------------------------------------------
TOKEN COUNT ANALYSIS FOR FAILED JSONS:
--------------------------------------------------
Total Failed JSONs Analyzed: 110
Number of outputs hitting the 1000 token limit: 98
Average tokens per failed output: 940.9
Max tokens seen: 1000
--------------------------------------------------
Observed Character-to-Token ratio for these failures: 1.77 chars/token
*(If this ratio is low, e.g., < 2.5, it proves numeric arrays tokenize highly inefficiently)*

Top 5 longest outputs (by token count):
1. Image ID: 0001715 | Tokens: 1000 | Chars: 1848 | Unclosed Brackets: True
2. Image ID: 0000938 | Tokens: 1000 | Chars: 1699 | Unclosed Brackets: True
3. Image ID: 0004303 | Tokens: 1000 | Chars: 1732 | Unclosed Brackets: True
4. Image ID: 0000666 | Tokens: 1000 | Chars: 1418 | Unclosed Brackets: True
5. Image ID: 0004249 | Tokens: 1000 | Chars: 1551 | Unclosed Bra

============================================================================

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_from_disk

DATASET_PATH = "/content/drive/MyDrive/vlm-finetuning-project1/datasets/processed"
assert os.path.exists(DATASET_PATH), "Dataset not found at this path!"

print("Loading dataset...")
dataset = load_from_disk(DATASET_PATH)
print(dataset)

Loading dataset...
DatasetDict({
    train: Dataset({
        features: ['image', 'image_id', 'image_caption', 'illumination', 'camera_distance', 'view', 'quality_of_info', 'rule_1_violation', 'rule_2_violation', 'rule_3_violation', 'rule_4_violation', 'excavator', 'rebar', 'worker_with_white_hard_hat', 'resolution'],
        num_rows: 6308
    })
    val: Dataset({
        features: ['image', 'image_id', 'image_caption', 'illumination', 'camera_distance', 'view', 'quality_of_info', 'rule_1_violation', 'rule_2_violation', 'rule_3_violation', 'rule_4_violation', 'excavator', 'rebar', 'worker_with_white_hard_hat', 'resolution'],
        num_rows: 701
    })
    test: Dataset({
        features: ['image', 'image_id', 'image_caption', 'illumination', 'camera_distance', 'view', 'quality_of_info', 'rule_1_violation', 'rule_2_violation', 'rule_3_violation', 'rule_4_violation', 'excavator', 'rebar', 'worker_with_white_hard_hat', 'resolution'],
        num_rows: 3004
    })
})


In [ ]:
all_stats = {}
PERCENTILES = [1, 5, 10, 25, 50, 75, 90, 95, 99]

for split in dataset.keys():
    res = np.array(dataset[split]["resolution"])

    # Pull width/height directly from the image objects (no .map, no schema change)
    widths = np.array([img.width for img in dataset[split]["image"]])
    heights = np.array([img.height for img in dataset[split]["image"]])
    aspect = widths / heights

    pcts = {f"p{p}": float(np.percentile(res, p)) for p in PERCENTILES}

    split_stats = {
        "count": len(res),
        "min_pixels": int(res.min()),
        "max_pixels": int(res.max()),
        "mean_pixels": float(res.mean()),
        "std_pixels": float(res.std()),
        "percentiles": pcts,
        "min_dims": f"{widths[res.argmin()]}x{heights[res.argmin()]}",
        "max_dims": f"{widths[res.argmax()]}x{heights[res.argmax()]}",
        "aspect_ratio_min": float(aspect.min()),
        "aspect_ratio_max": float(aspect.max()),
        "aspect_ratio_mean": float(aspect.mean()),
    }
    all_stats[split] = split_stats

    print(f"\n=== {split.upper()} ({len(res)} images) ===")
    print(f"Min:  {res.min():>12,} px  ({split_stats['min_dims']})")
    print(f"Max:  {res.max():>12,} px  ({split_stats['max_dims']})")
    print(f"Mean: {res.mean():>12,.0f} px   Std: {res.std():,.0f}")
    for p in PERCENTILES:
        print(f"  P{p:<3}: {pcts[f'p{p}']:>12,.0f} px")
    print(f"Aspect ratio range: {aspect.min():.2f} - {aspect.max():.2f} (mean {aspect.mean():.2f})")


=== TRAIN (6308 images) ===
Min:        47,232 px  (288x164)
Max:    14,625,792 px  (3312x4416)
Mean:    1,120,056 px   Std: 1,113,808
  P1  :      348,771 px
  P5  :      414,720 px
  P10 :      693,090 px
  P25 :      939,048 px
  P50 :    1,080,000 px
  P75 :    1,080,000 px
  P90 :    1,080,000 px
  P95 :    1,228,800 px
  P99 :    4,915,200 px
Aspect ratio range: 0.45 - 5.20 (mean 1.31)

=== VAL (701 images) ===
Min:       230,560 px  (524x440)
Max:    14,625,792 px  (4416x3312)
Mean:    1,216,896 px   Std: 1,506,937
  P1  :      388,800 px
  P5  :      414,720 px
  P10 :      693,090 px
  P25 :      843,936 px
  P50 :    1,080,000 px
  P75 :    1,080,000 px
  P90 :    1,080,000 px
  P95 :    1,555,200 px
  P99 :   11,808,768 px
Aspect ratio range: 0.56 - 4.55 (mean 1.32)

=== TEST (3004 images) ===
Min:        76,800 px  (320x240)
Max:    14,625,792 px  (4416x3312)
Mean:    1,157,066 px   Std: 1,132,906
  P1  :      388,800 px
  P5  :      688,892 px
  P10 :      750,000 px
  P2

In [ ]:
BUCKETS = [
    (0, 100_000, "tiny (<100K px)"),
    (100_000, 500_000, "small (100K-500K)"),
    (500_000, 1_000_000, "medium (500K-1M)"),
    (1_000_000, 1_200_000, "standard (1M-1.2M) <- bulk of dataset"),
    (1_200_000, 3_000_000, "large (1.2M-3M)"),
    (3_000_000, 8_000_000, "very large (3M-8M)"),
    (8_000_000, float("inf"), "extreme outlier (8M+)"),
]

for split in dataset.keys():
    res = np.array(dataset[split]["resolution"])
    total = len(res)
    print(f"\n=== {split.upper()} BUCKET DISTRIBUTION ===")
    for lo, hi, label in BUCKETS:
        mask = (res >= lo) & (res < hi)
        count = mask.sum()
        pct = count / total * 100
        # find index range of this bucket ASSUMING the split is sorted by resolution
        idxs = np.where(mask)[0]
        idx_range = f"[{idxs.min()} to {idxs.max()}]" if len(idxs) else "none"
        print(f"  {label.ljust(38)}: {count:>5} imgs ({pct:5.1f}%)  index range {idx_range}")


=== TRAIN BUCKET DISTRIBUTION ===
  tiny (<100K px)                       :    13 imgs (  0.2%)  index range [0 to 12]
  small (100K-500K)                     :   344 imgs (  5.5%)  index range [13 to 356]
  medium (500K-1M)                      :  1570 imgs ( 24.9%)  index range [357 to 1926]
  standard (1M-1.2M) <- bulk of dataset :  3936 imgs ( 62.4%)  index range [1927 to 5862]
  large (1.2M-3M)                       :   315 imgs (  5.0%)  index range [5863 to 6177]
  very large (3M-8M)                    :    76 imgs (  1.2%)  index range [6178 to 6253]
  extreme outlier (8M+)                 :    54 imgs (  0.9%)  index range [6254 to 6307]

=== VAL BUCKET DISTRIBUTION ===
  tiny (<100K px)                       :     0 imgs (  0.0%)  index range none
  small (100K-500K)                     :    40 imgs (  5.7%)  index range [0 to 39]
  medium (500K-1M)                      :   186 imgs ( 26.5%)  index range [40 to 225]
  standard (1M-1.2M) <- bulk of dataset :   422 imgs ( 60.2

# Clean Corrupted/Empty Records from predictions.jsonl

In [9]:
import os
import json
import shutil
from pathlib import Path
from datetime import datetime

MODEL_TIER = "2b"
# DRIVE_RESULTS_DIR = Path(f"/content/drive/MyDrive/vlm-finetuning-project1/results/baseline_test_full_{MODEL_TIER}")
# JSONL_OUTPUT_PATH = str(DRIVE_RESULTS_DIR / "predictions.jsonl")

assert os.path.exists(JSONL_OUTPUT_PATH), "File not found!"

In [ ]:
from datasets import load_from_disk

DATASET_PATH = "/content/drive/MyDrive/vlm-finetuning-project1/datasets/processed"
dataset = load_from_disk(DATASET_PATH)
EXPECTED_COUNT = len(dataset["test"])
print(f"Expected total records (test split size): {EXPECTED_COUNT}")

Expected total records (test split size): 3004


In [10]:
cleaned_records = []
seen_image_ids = set()
duplicates = []
empty_count = 0
malformed_count = 0

with open(JSONL_OUTPUT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            record = json.loads(line)
        except json.JSONDecodeError:
            malformed_count += 1
            continue

        raw_output = record.get("raw_output", "")
        img_id = record.get("image_id")

        if not raw_output.strip():
            empty_count += 1
            continue

        if img_id in seen_image_ids:
            duplicates.append(img_id)
            continue  # keep the FIRST occurrence, drop later duplicates

        seen_image_ids.add(img_id)
        cleaned_records.append(record)

print(f"Malformed rows dropped:     {malformed_count}")
print(f"Empty/crashed rows dropped: {empty_count}")
print(f"Duplicate rows dropped:     {len(duplicates)}")
print(f"Clean unique records kept:  {len(cleaned_records)}")

Malformed rows dropped:     0
Empty/crashed rows dropped: 0
Duplicate rows dropped:     0
Clean unique records kept:  3004


In [11]:
all_expected_ids = set(str(x) for x in dataset["test"]["image_id"])
found_ids = set(str(x) for x in seen_image_ids)
missing_ids = sorted(all_expected_ids - found_ids)

print(f"Missing image_ids: {len(missing_ids)}")
if missing_ids:
    print(f"First 20 missing: {missing_ids[:20]}")

NameError: name 'dataset' is not defined

In [ ]:
if len(cleaned_records) == EXPECTED_COUNT:
    print("Counts match exactly. Overwriting predictions.jsonl with cleaned data...")
    with open(JSONL_OUTPUT_PATH, "w", encoding="utf-8") as f:
        for record in cleaned_records:
            f.write(json.dumps(record) + "\n")
    print("✅ SUCCESS! predictions.jsonl is now clean, deduplicated, and ready for evaluation.")
    print(f"Backup remains at: {backup_path} (safe to delete once you confirm everything looks right)")
else:
    print(f"⚠️ Count mismatch: got {len(cleaned_records)}, expected {EXPECTED_COUNT}.")
    print("File was NOT overwritten. Options:")
    print("  1. Re-run inference for the missing_image_ids list, then re-run this task.")
    print("  2. If this is expected (e.g. --max_samples was used), manually confirm and overwrite yourself.")